# 04 · Multi-step CreativeIR decompiler review

Two-pass pipeline (Pass A factual shot analysis → Pass B global creative synthesis) compared against the #4 single-pass baseline.
All logic lives in `src/tiktok_analytics_factory/multistep/`; this notebook only displays persisted run artifacts.

In [ ]:
from pathlib import Path
import json, pandas as pd

VIDEO_ID = "6718335390845095173"
RUN_DIR = sorted((Path("data/derived") / VIDEO_ID / "decompilation" / "multi_step").glob("*"))[-1]
PERCEPTION = Path("data/derived") / VIDEO_ID / "perception" / "v1"
print("Run:", RUN_DIR)

## Source video & deterministic perception

In [ ]:
from IPython.display import Video
Video(str(Path("data/raw") / VIDEO_ID / "video.mp4"))

In [ ]:
shots = json.loads((RUN_DIR / "shots.json").read_text())
pd.DataFrame(shots["shots"])

### Representative frames per shot

In [ ]:
from IPython.display import Image
for f in json.loads((PERCEPTION / "perception_manifest.json").read_text())["frames"]:
    print(f["shot_id"], f"t={f['timestamp_seconds']}s")
    display(Image(filename=f["path"], width=180))

## Final CreativeIR

In [ ]:
creative_ir = json.loads((RUN_DIR / "creative_ir.json").read_text())
creative_ir["decompilation"]

In [ ]:
pd.DataFrame(creative_ir["observed"]["shots"]).drop(columns=["evidence"])

In [ ]:
pd.DataFrame([creative_ir["inferred"]["concept"],
              creative_ir["inferred"]["target_audience_hypothesis"],
              creative_ir["inferred"]["hook_type"],
              creative_ir["inferred"]["narrative_structure"]])

In [ ]:
json.loads((RUN_DIR / "validation.json").read_text())

## Single-pass vs multi-step scores (same 11-category rubric)

In [ ]:
comparison = json.loads((RUN_DIR / "comparison_vs_single_pass.json").read_text())
rows = [{"category": r["category"], "single_pass": r["single_pass_score"],
         "multi_step": r["multi_step_score"], "delta": r["delta"]} for r in comparison["categories"]]
df = pd.DataFrame(rows)
df.loc[len(df)] = {"category": "OVERALL AVG", 
                   "single_pass": comparison["overall_average"]["single_pass"],
                   "multi_step": comparison["overall_average"]["multi_step"], "delta": None}
df

## Unsupported claims & material omissions

In [ ]:
u = comparison["unsupported_claims"]
pd.DataFrame([{**c, "pipeline": "multi_step"} for c in u["multi_step"]] +
             [{**c, "pipeline": "single_pass"} for c in u["single_pass"]])

In [ ]:
pd.Series({"single_pass_omissions": comparison["material_omissions"]["single_pass_count"],
           "multi_step_omissions": comparison["material_omissions"]["multi_step_count"]})

## Cost / latency breakdown

In [ ]:
usage = json.loads((RUN_DIR / "usage.json").read_text())
pd.DataFrame(usage["passes"])[["pass", "model_id", "prompt_version", "latency_seconds", "cost_usd"]]

In [ ]:
pd.Series({k: usage[k] for k in ("model_call_count", "total_usage", "total_cost_usd", "total_latency_seconds")})